In [36]:
import re
from docx import Document
import pandas as pd

# === 1. Đọc nội dung Word ===
def extract_text_from_docx(path):
    doc = Document(path)
    paragraphs = []
    for para in doc.paragraphs:
        text = para.text.strip()
        if text:
            paragraphs.append(text)
    return "\n".join(paragraphs), doc


# === 2. Tách phần đề và phần đáp án ===
def split_sections(text):
    marker = "ĐÁP ÁN VÀ HƯỚNG DẪN GIẢI CHI TIẾT"
    if marker in text:
        idx = text.find(marker)
        return text[:idx], text[idx:]
    return text, ""


# === 3. Tách câu hỏi và các lựa chọn (A–D) ===
def extract_questions(de_text):
    """
    Trích danh sách câu hỏi từ phần đề, chỉ lấy nội dung câu hỏi (không lấy lựa chọn A–D)
    """
    # regex tìm từng khối câu hỏi: "Câu 1: ...", "Câu 2: ..."
    pattern = r"Câu\s+(\d+):([\s\S]*?)(?=\nCâu\s+\d+:|PHẦN|$)"
    matches = re.findall(pattern, de_text)
    
    questions = []
    for num, qtext in matches:
        # làm sạch xuống dòng và khoảng trắng  
        questions.append({
            "id": int(num),
            "question_text": qtext
        })
    return questions


def extract_answers_from_docx(doc):
    import re

    answers = []
    pattern = r"Câu\s+(\d+)"
    current_id = None
    current_text = []
    current_underline = None
    inside_answer_section = False

    for para in doc.paragraphs:
        text = para.text.strip()
        if not text:
            continue

        # Bắt đầu đọc khi gặp đoạn "ĐÁP ÁN VÀ HƯỚNG DẪN GIẢI"
        if not inside_answer_section:
            if "ĐÁP ÁN" in text.upper() and "HƯỚNG DẪN" in text.upper():
                inside_answer_section = True
            continue

        # Kiểm tra có phải đoạn numbering (Câu mới)
        is_numbered = para._p.pPr is not None and para._p.pPr.numPr is not None
        m = re.match(pattern, text)

        # Nếu là đoạn bắt đầu câu hỏi mới
        if m or is_numbered:
            if current_id is not None:
                # Lấy hướng dẫn giải 
                full_text = "\n".join(current_text).strip()
                match = re.search(r"hướng\s*dẫn\s*giải[:\-]?", full_text, re.IGNORECASE)
                if match:
                    full_text = full_text[match.end():].strip()  # lấy sau cụm đó

                final = "Đáp án: " + (current_underline or "") + "\n" + (full_text or "")

                answers.append({
                    "id": current_id,
                    "answer": final
                })

            if m:
                current_id = int(m.group(1))
            else:
                current_id = (current_id or 0) + 1

            current_text = []
            current_underline = None
            continue

        # Kiểm tra đáp án
        underline_letters = []
        for run in para.runs:
            if run.font.underline and re.match(r"^[A-D]\.?$", run.text.strip()):
                underline_letters.append(run.text.strip().replace('.', ''))
        if underline_letters:
            current_underline = underline_letters[0]

        current_text.append(text)

    # Lưu câu cuối cùng
    if current_id is not None:
        full_text = "\n".join(current_text).strip()
        match = re.search(r"hướng\s*dẫn\s*giải[:\-]?", full_text, re.IGNORECASE)
        if match:
            full_text = full_text[match.end():].strip()

        final = "Đáp án: " + (current_underline or "") + "\n" + (full_text or "")

        answers.append({
            "id": current_id,
            "answer": final
        })

    return answers


# === 5. Gộp câu hỏi và đáp án ===
def merge_q_and_a(questions, answers):
    merged = []
    for q in questions:
        ans = next((a for a in answers if a["id"] == q["id"]), None)
        merged.append({
            "id": q["id"],
            "question_text": q["question_text"],
            "answer": ans["answer"] if ans else None,
        })
    return merged


# === 6. Chạy thử ===
docx_path = "test_data/01_TdtntTayNguyen_Ntt.docx"  # file Word của bạn
text, doc = extract_text_from_docx(docx_path)
de, dap_an_text = split_sections(text)
questions = extract_questions(de)
answers = extract_answers_from_docx(doc)
merged_data = merge_q_and_a(questions, answers)

# Xuất kết quả
df = pd.DataFrame(merged_data)
print(df.head())

df.to_excel("output_docx_cauhoi_dapan.xlsx", index=False)
print("✅ Đã lưu file output_docx_cauhoi_dapan.xlsx")


   id                                      question_text  \
0   1   Trong điều kiện chuẩn về nhiệt độ và áp suất ...   
1   2   Khi truyền nhiệt cho một khối khí thì khối kh...   
2   3   Câu nào sau đây nói về nhiệt lượng là không đ...   
3   4   Phóng xạ là\nA. quá trình hạt nhân nguyên tử ...   
4   5   Phát biểu nào sau đây là không đúng khi nói v...   

                                              answer  
0  Đáp án: A\nTrong điều kiện chuẩn về nhiệt độ v...  
1  Đáp án: A\nKhi truyền nhiệt cho một khối khí t...  
2  Đáp án: B\nNhiệt lượng là số đo độ biến thiên ...  
3  Đáp án: B\n- Phóng xạ là hiện tượng hạt nhân n...  
4  Đáp án: B\nĐiện trường xoáy có các đường sức l...  
✅ Đã lưu file output_docx_cauhoi_dapan.xlsx


In [21]:
from docx import Document

def extract_underlined_words(docx_path):
    doc = Document(docx_path)
    underlined_words = []

    for para in doc.paragraphs:
        for run in para.runs:
            if run.font.underline:  # kiểm tra gạch chân
                text = run.text.strip()
                if text:
                    underlined_words.append(text)
    return underlined_words


# --- Sử dụng ---
file_path = "01_TdtntTayNguyen_Ntt.docx"
underlined = extract_underlined_words(file_path)

print("Các từ được gạch chân:")
for word in underlined:
    print(word)


Các từ được gạch chân:
A
A
B
B
B
A
A
A
B
C
D
A
B
D
B
D
B
B
